# VASU V4 — gated, onset-aligned DS-CNN

One bounded correction after V3 showed high confusable-word false activation. Uses speakers 1–3 only; speakers 4 and 5 remain sealed.

## 1 — Upload and reproduce the reviewed dataset

In [ ]:
import os, sys, json, io, zipfile, hashlib, re, shutil, time, importlib.util
from pathlib import Path
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '2')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '2')
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.fft import rfft
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 1729
SMOKE = os.environ.get('VASU_SMOKE', '0') == '1'  # Local engineering check only.
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass
BASE = Path(os.environ.get('VASU_BASE', '/content'))
RUN = BASE / ('vasu_v4_smoke' if SMOKE else 'vasu_training_v4')
RUN.mkdir(parents=True, exist_ok=True)
SR = 16000
# Preserve the complete 1.2-second context that was manually reviewed.
# This deliberately replaces the OLD 1-second / 49-frame frontend.
N = 19200
FRAME, HOP, FFT, MELS = 480, 320, 512, 40
FRAMES = 1 + (N - FRAME) // HOP  # 59
LABELS = ['vasu', 'confusable', 'other_speech', 'background']
EPOCHS = 2 if SMOKE else 32
THRESHOLDS = np.array([.35, .45, .55, .65, .75, .85, .90, .94, .97, .99])
print('TensorFlow:', tf.__version__, '| Python:', sys.version.split()[0])
print('GPU:', bool(tf.config.list_physical_devices('GPU')))
print('Model input:', (FRAMES, MELS, 1), '| duration: 1.2 seconds')
print('Gated 3-class VASU training. Speakers 4 and 5 sealed. No pretrained KWS model.')
assert FRAMES == 59

required_zips = ['recordings.zip', 'vasu_dataset_audit_checkpoint_v1.zip']
if not all((BASE / f).exists() for f in required_zips):
    from google.colab import files
    print('Select BOTH recordings.zip and vasu_dataset_audit_checkpoint_v1.zip')
    files.upload()
assert all((BASE / f).exists() for f in required_zips), 'Both ZIP files are required.'

def read_csv(z, name):
    matches = [p for p in z.namelist() if Path(p).name == name]
    assert len(matches) == 1, f'Ambiguous/missing checkpoint file: {name}'
    return pd.read_csv(z.open(matches[0])).fillna('')

with zipfile.ZipFile(BASE / required_zips[1]) as z:
    manifest = read_csv(z, 'vasu_dataset_manifest_v1.csv')
    words = read_csv(z, 'accepted_word_candidates_v1.csv')
    rejected = read_csv(z, 'rejected_word_candidates_v1.csv')
    negatives = read_csv(z, 'negative_windows_final_v1.csv')
assert len(words) == 415 and len(negatives) == 1197
assert not words.speaker.eq('speaker04').any()
assert not negatives.speaker.eq('speaker04').any()
active = manifest[manifest.split.isin(['train', 'development'])].copy()
assert not active.speaker.eq('speaker04').any()
assert active.filename.is_unique
assert manifest.groupby('sha256').split.nunique().max() == 1
sources = active.set_index('filename').to_dict('index')
audio = {}
with zipfile.ZipFile(BASE / required_zips[0]) as z:
    names = z.namelist()
    for row in active.itertuples():
        matches = [p for p in names if Path(p).name == row.filename]
        assert len(matches) == 1, f'Missing/duplicate WAV: {row.filename}'
        raw = z.read(matches[0])  # No speaker04 bytes are read.
        assert hashlib.sha256(raw).hexdigest() == row.sha256, f'Changed WAV: {row.filename}'
        rate, x = wavfile.read(io.BytesIO(raw))
        assert rate == SR and x.ndim == 1 and x.dtype == np.int16
        audio[row.filename] = x

def bounds(filename, include_rejected=True):
    result = []
    text = sources[filename]['exclusions']
    for a, b in re.findall(r'(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)', str(text)):
        result.append((max(0, float(a) - .25), float(b) + .25))
    # Previously declared manual exclusions from the long-candidate review.
    extra = {
        'spk02_dev_kw_mixed_s05.wav': [(22.3, 25.9)],
        'spk03_train_confusable_s01.wav': [(83.8, 86.5)],
        'spk02_train_kw_normal_s01.wav': [(45.8, 46.9)],
    }
    result.extend(extra.get(filename, []))
    if include_rejected:
        for row in rejected[rejected.filename.eq(filename)].itertuples():
            result.append((float(row.review_start_s), float(row.review_end_s)))
    return result

def safe_interval(filename, start, end):
    return not any(start < b and end > a for a, b in bounds(filename))

def extract(filename, start):
    x = audio[filename]
    a = int(round(start * SR))
    assert a >= 0 and a + N <= len(x), (filename, start)
    return x[a:a + N].astype(np.float32) / 32768.0

items, omitted = [], []
for row in words.itertuples():
    start = float(row.review_start_s)
    end = start + N / SR
    if abs(float(row.review_end_s) - end) >= .002:
        omitted.append({'filename': row.filename, 'start_s': start, 'reason': 'short_boundary_review_clip', 'id': row.candidate_id})
        continue
    if start < 0 or end > len(audio[row.filename]) / SR or not safe_interval(row.filename, start, end):
        omitted.append({'filename': row.filename, 'start_s': start, 'reason': 'boundary_or_exclusion_overlap', 'id': row.candidate_id})
        continue
    items.append({'filename': row.filename, 'speaker': row.speaker, 'split': row.split,
                  'label': row.proposed_label, 'start_s': start, 'end_s': end, 'id': row.candidate_id})
for row in negatives.itertuples():
    # Extend reviewed 1s negative windows by 0.1s on each side using ORIGINAL audio.
    # Clamp to recording boundaries, then recheck exclusions over the ENTIRE new window.
    duration = len(audio[row.filename]) / SR
    start = max(0., min(float(row.start_s) - .1, duration - N / SR))
    end = start + N / SR
    if not safe_interval(row.filename, start, end):
        omitted.append({'filename': row.filename, 'start_s': start, 'reason': 'expanded_negative_exclusion', 'id': row.window_id})
        continue
    items.append({'filename': row.filename, 'speaker': row.speaker, 'split': row.split,
                  'label': row.model_label, 'start_s': start, 'end_s': end, 'id': row.window_id})
items = pd.DataFrame(items).drop_duplicates(['filename', 'start_s', 'label']).reset_index(drop=True)
assert items.groupby('filename').split.nunique().max() == 1
assert not items.speaker.eq('speaker04').any()
items.to_csv(RUN / 'used_examples.csv', index=False)
pd.DataFrame(omitted).to_csv(RUN / 'excluded_context_windows.csv', index=False)
print('Loaded', len(audio), 'train/development recordings; 0 test recordings.')
print(items.groupby(['split', 'label']).size().to_string())
print('Context/exclusion checks removed:', len(omitted), '(logged, not silently relabeled)')
print('No additional human listening is required for this preparation step.')

# PCM -> per-frame mean removal -> periodic Hann -> RFFT -> power -> HTK Mel -> ln.
# No audio-library defaults, no per-clip peak normalization, no test-derived statistics.
window = (0.5 - 0.5 * np.cos(2 * np.pi * np.arange(FRAME) / FRAME)).astype(np.float32)
hz_to_mel = lambda f: 2595.0 * np.log10(1 + f / 700.)
mel_to_hz = lambda m: 700. * (10 ** (m / 2595.) - 1)
edges = mel_to_hz(np.linspace(hz_to_mel(80.), hz_to_mel(7600.), MELS + 2))
freq = np.arange(FFT // 2 + 1) * SR / FFT
mel = np.maximum(0., np.minimum((freq[:, None] - edges[:-2]) / (edges[1:-1] - edges[:-2]),
                              (edges[2:] - freq[:, None]) / (edges[2:] - edges[1:-1]))).astype(np.float32)

def frontend(x):
    x = np.asarray(x, np.float32)
    assert x.shape[-1] == N
    f = np.lib.stride_tricks.sliding_window_view(x, FRAME, axis=-1)[..., ::HOP, :]
    f = (f - f.mean(axis=-1, keepdims=True)) * window
    spectrum = rfft(f, n=FFT, axis=-1)
    power = (spectrum.real ** 2 + spectrum.imag ** 2) / float(FFT * FFT)
    return np.log(np.maximum(power @ mel, 1e-12)).astype(np.float32)

rng = np.random.default_rng(SEED)
train_items = items[items.split.eq('train')].reset_index(drop=True)
dev_items = items[items.split.eq('development')].reset_index(drop=True)
noise_sources = [f for f, r in sources.items() if r['split'] == 'train' and r['recording_category'] == 'background']
assert noise_sources

def augment(x):
    y = x.copy()
    # Only shift when the samples that would be dropped are quiet relative to the clip.
    shift = int(rng.integers(-1600, 1601))
    if shift:
        edge = y[-shift:] if shift > 0 else y[:-shift]
        if np.sqrt(np.mean(edge ** 2) + 1e-12) < .15 * np.sqrt(np.mean(y ** 2) + 1e-12):
            fill = np.zeros_like(y)
            if shift > 0:
                fill[shift:] = y[:-shift]
            else:
                fill[:shift] = y[-shift:]
            y = fill
    y *= 10 ** (rng.uniform(-6., 4.) / 20.)
    if rng.random() < .65:
        source = audio[rng.choice(noise_sources)].astype(np.float32) / 32768.
        start = int(rng.integers(0, len(source) - N + 1))
        n = source[start:start + N].copy()
        n -= n.mean()
        signal_rms = np.sqrt(np.mean(y ** 2) + 1e-12)
        noise_rms = np.sqrt(np.mean(n ** 2) + 1e-12)
        y += n * signal_rms / (noise_rms * 10 ** (rng.uniform(12., 30.) / 20.))
    peak = np.max(np.abs(y))
    if peak > .98:
        y *= .98 / peak
    return y

train_wave = np.stack([extract(r.filename, r.start_s) for r in train_items.itertuples()])
dev_wave = np.stack([extract(r.filename, r.start_s) for r in dev_items.itertuples()])
base_features = frontend(train_wave)
mean = base_features.mean(axis=(0, 1)).astype(np.float32)
std = np.maximum(base_features.std(axis=(0, 1)), .5).astype(np.float32)
def normalize(f):
    return np.clip((f - mean) / std, -8., 8.).astype(np.float32)[..., None]

X_train_parts = [normalize(base_features)]
for repetition in range(1 if SMOKE else 4):
    X_train_parts.append(normalize(frontend(np.stack([augment(x) for x in train_wave]))))
X_train = np.concatenate(X_train_parts)
X_dev = normalize(frontend(dev_wave))
y_base = np.array([LABELS.index(x) for x in train_items.label], dtype=np.int32)
y_train = np.tile(y_base, len(X_train_parts))
y_dev = np.array([LABELS.index(x) for x in dev_items.label], dtype=np.int32)
# Equal total weight per class, then per represented speaker within a class.
weights = np.empty(len(train_items), np.float32)
for label, group in train_items.groupby('label'):
    counts = group.speaker.value_counts()
    for idx, row in group.iterrows():
        weights[idx] = 1. / (len(LABELS) * len(counts) * counts[row.speaker])
weights /= weights.mean()
sample_weights = np.tile(weights, len(X_train_parts))
np.savez(RUN / 'frontend_constants.npz', hann=window, mel=mel, mean=mean, std=std)
frontend_config = dict(sample_rate=SR, pcm_scale=32768., audio_samples=N, input_shape=[FRAMES, MELS, 1],
                      frame_samples=FRAME, hop_samples=HOP, fft_size=FFT, mel_bins=MELS,
                      mel_low_hz=80., mel_high_hz=7600., mel_formula='HTK; triangles on FFT-bin frequencies; no area normalization',
                      window='periodic Hann', dc_removal='per-frame mean, before Hann',
                      power_divisor=FFT*FFT, log='natural', log_floor=1e-12,
                      normalization='training-only per-Mel mean/std; clip [-8,8]',
                      frame_padding=False, labels=LABELS, inference_hop_samples=1600)
(RUN / 'frontend.json').write_text(json.dumps(frontend_config, indent=2))
assert X_train.shape[1:] == (59, 40, 1) and np.isfinite(X_train).all()
print('Original training windows:', len(train_items), '| including variations:', len(X_train))
print('Development windows:', len(X_dev), '| shape:', X_dev.shape[1:])
print('Variations are NOT extra speakers or independent recordings.')

## 2 — Train, select, quantize and audit

In [ ]:

# V4: align every reviewed utterance to its acoustic onset and explicitly learn
# Vasu vs confusable words vs other audio. This is still development work only.
Y3_base = np.where(y_base == 0, 0, np.where(y_base == 1, 1, 2)).astype(np.int32)
Y3_dev = np.where(y_dev == 0, 0, np.where(y_dev == 1, 1, 2)).astype(np.int32)

VAD_FRAME = 320       # 20 ms
VAD_HOP = 160         # 10 ms
VAD_PRE = 2880        # 180 ms of pre-roll
VAD_START_FRAMES = 3  # 30 ms attack confirmation
VAD_END_FRAMES = 12   # 120 ms hangover
VAD_FACTOR = 2.0
VAD_MIN_RMS = 0.0012

def rms_track(x):
    f = np.lib.stride_tricks.sliding_window_view(
        np.asarray(x, np.float32), VAD_FRAME)[::VAD_HOP]
    return np.sqrt(np.mean(f*f, axis=1) + 1e-12)

def dominant_onset(x):
    """Find the start of the dominant speech burst inside one reviewed window."""
    rms = rms_track(x)
    floor = max(float(np.percentile(rms, 20)), VAD_MIN_RMS)
    high = float(np.percentile(rms, 95))
    threshold = max(floor * VAD_FACTOR, floor + .12 * max(0., high-floor))
    active = rms >= threshold
    # Bridge short holes, then choose the active group containing the loudest frame.
    for i in range(1, len(active)-1):
        if not active[i] and active[i-1] and active[i+1]: active[i] = True
    peak = int(np.argmax(rms))
    if not active.any(): return max(0, peak*VAD_HOP - VAD_PRE)
    if not active[peak]: peak = int(np.flatnonzero(active)[np.argmax(rms[active])])
    left = peak
    gap = 0
    while left > 0:
        left -= 1
        gap = 0 if active[left] else gap + 1
        if gap > 5:
            left += gap
            break
    return max(0, int(left*VAD_HOP) - VAD_PRE)

def onset_align(x):
    start = dominant_onset(x)
    result = np.zeros(N, np.float32)
    source = np.asarray(x, np.float32)
    available = min(N, len(source)-start)
    result[:available] = source[start:start+available]
    # dominant_onset returns pre-roll position; preserve original noise rather than
    # peak-normalising, so the device frontend sees the same amplitude convention.
    return result

aligned_train = np.stack([onset_align(x) for x in train_wave])
aligned_dev = np.stack([onset_align(x) for x in dev_wave])

# Four genuinely different variations. Time shift remains small because onset is
# already normalised; gain/noise augmentation remains training-only.
rng_v4 = np.random.default_rng(SEED + 44)
def augment_v4(x):
    y = x.copy()
    shift = int(rng_v4.integers(-800, 801))
    if shift > 0:
        y = np.pad(y[:-shift], (shift, 0))
    elif shift < 0:
        y = np.pad(y[-shift:], (0, -shift))
    y *= 10 ** (rng_v4.uniform(-5., 4.) / 20.)
    if rng_v4.random() < .70:
        source = audio[rng_v4.choice(noise_sources)].astype(np.float32) / 32768.
        a = int(rng_v4.integers(0, len(source)-N+1))
        n = source[a:a+N].copy(); n -= n.mean()
        srms = np.sqrt(np.mean(y*y)+1e-12)
        nrms = np.sqrt(np.mean(n*n)+1e-12)
        y += n * srms / (nrms * 10**(rng_v4.uniform(10., 28.)/20.))
    peak = np.max(np.abs(y))
    if peak > .98: y *= .98/peak
    return y.astype(np.float32)

train_waves = [aligned_train]
for _ in range(1 if SMOKE else 5):
    train_waves.append(np.stack([augment_v4(x) for x in aligned_train]))
V4_train_wave = np.concatenate(train_waves)
V4_y = np.tile(Y3_base, len(train_waves))

# Extra confusable variations give the first/last phonemes equal influence to Vasu.
conf_idx = np.flatnonzero(Y3_base == 1)
extra_conf = np.stack([augment_v4(aligned_train[i]) for i in conf_idx])
V4_train_wave = np.concatenate([V4_train_wave, extra_conf])
V4_y = np.concatenate([V4_y, np.ones(len(extra_conf), np.int32)])

raw_v4 = frontend(V4_train_wave)
v4_mean = raw_v4.mean(axis=(0,1)).astype(np.float32)
v4_std = np.maximum(raw_v4.std(axis=(0,1)), .5).astype(np.float32)
def normalize_v4(f):
    return np.clip((f-v4_mean)/v4_std, -8., 8.).astype(np.float32)[...,None]
V4_X = normalize_v4(raw_v4)
V4_X_dev = normalize_v4(frontend(aligned_dev))

def balanced3(y):
    y = np.asarray(y)
    w = np.zeros(len(y), np.float32)
    for label in range(3):
        idx = np.flatnonzero(y == label)
        w[idx] = 1.0/max(1, len(idx))
    return w/w.mean()
V4_W = balanced3(V4_y)
V4_W_dev = balanced3(Y3_dev)

reg = tf.keras.regularizers.l2(1.5e-4)
def ds(x, channels, stride=(1,1), kernel=(5,3)):
    x = tf.keras.layers.DepthwiseConv2D(
        kernel, strides=stride, padding='same', use_bias=False,
        depthwise_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization(momentum=.9)(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Conv2D(
        channels, 1, use_bias=False, kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization(momentum=.9)(x)
    return tf.keras.layers.ReLU()(x)

def build_v4():
    inp = tf.keras.Input((FRAMES, MELS, 1))
    # Preserve onset/phoneme timing: frequency is reduced before time.
    x = tf.keras.layers.Conv2D(
        12, (5,3), strides=(1,2), padding='same', use_bias=False,
        kernel_regularizer=reg)(inp)
    x = tf.keras.layers.BatchNormalization(momentum=.9)(x)
    x = tf.keras.layers.ReLU()(x)
    x = ds(x, 20, stride=(2,2))
    x = ds(x, 28, stride=(1,2))
    x = ds(x, 32, stride=(1,1), kernel=(5,1))
    x = tf.keras.layers.AveragePooling2D((1, int(x.shape[2])))(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dropout(.35)(x)
    x = tf.keras.layers.Dense(20, activation='relu', kernel_regularizer=reg)(x)
    out = tf.keras.layers.Dense(3, activation='softmax')(x)
    return tf.keras.Model(inp, out, name='vasu_gated_confusable_dscnn')

def predict3(model, X, batch=128):
    return np.concatenate([
        model(X[i:i+batch], training=False).numpy()
        for i in range(0, len(X), batch)])

def clip_v4(p, threshold=.5, margin=.15):
    hit = (p[:,0] >= threshold) & ((p[:,0]-p[:,1]) >= margin)
    pos = Y3_dev == 0
    recalls=[]
    for speaker in ('speaker01','speaker02','speaker03'):
        m = pos & dev_items.speaker.eq(speaker).to_numpy()
        recalls.append(float(hit[m].mean()) if m.any() else 0.)
    return dict(recall=float(hit[pos].mean()), worst_speaker_recall=min(recalls),
                confusable_false=float(hit[Y3_dev==1].mean()),
                other_false=float(hit[Y3_dev==2].mean()))

class KeepV4(tf.keras.callbacks.Callback):
    def __init__(self): self.rows=[]; self.saved=[]
    def on_epoch_end(self, epoch, logs=None):
        p=predict3(self.model,V4_X_dev)
        choices=[]
        for t in (.35,.45,.55,.65,.75,.85,.92):
            for m in (.0,.10,.20,.30,.40):
                q=clip_v4(p,t,m)
                good=q['recall']>=.90 and q['worst_speaker_recall']>=.80
                rank=(0 if good else 1,
                      2*q['confusable_false']+q['other_false'],
                      -q['recall'],-q['worst_speaker_recall'])
                choices.append((rank,t,m,q))
        rank,t,m,q=min(choices,key=lambda z:z[0])
        row=dict(epoch=epoch+1,threshold=t,margin=m,**q,
                 loss=float(logs['loss']),val_loss=float(logs['val_loss']))
        self.rows.append(row)
        path=RUN/f'v4_epoch_{epoch+1:02d}.keras'
        if len(self.saved)<6 or rank<max(x[0] for x in self.saved):
            self.model.save(path); self.saved.append((rank,str(path),row))
            self.saved.sort(key=lambda x:x[0]); self.saved=self.saved[:6]
        print(f"epoch {epoch+1}: recall={q['recall']:.1%}, "
              f"confusable false={q['confusable_false']:.1%}, t={t:.2f}, margin={m:.2f}")

tf.keras.utils.set_random_seed(SEED+4)
model=build_v4()
model.compile(optimizer=tf.keras.optimizers.Adam(.001,clipnorm=1.),
              loss='sparse_categorical_crossentropy')
keeper=KeepV4()
model.fit(V4_X,V4_y,sample_weight=V4_W,
          validation_data=(V4_X_dev,Y3_dev,V4_W_dev),
          batch_size=32,epochs=(2 if SMOKE else 35),verbose=0,
          callbacks=[keeper,
                     tf.keras.callbacks.ReduceLROnPlateau(
                         monitor='val_loss',patience=3,factor=.5,min_lr=2e-5),
                     tf.keras.callbacks.EarlyStopping(
                         monitor='val_loss',patience=9)])
pd.DataFrame(keeper.rows).to_csv(RUN/'training_history.csv',index=False)
print('V4 parameters:',model.count_params())

# Exact lightweight sequential activity gate used for continuous development.
def gate_candidates(pcm):
    x=np.asarray(pcm,np.float32)/32768.
    rms=rms_track(x)
    init=rms[:min(len(rms),50)]
    noise=max(float(np.percentile(init,25)),VAD_MIN_RMS)
    active_run=quiet_run=0; in_speech=False; onset=0; last_emit=-10**9
    candidates=[]
    for i,e in enumerate(rms):
        threshold=max(VAD_MIN_RMS,noise*VAD_FACTOR)
        active=e>=threshold
        if not in_speech and not active:
            noise=.995*noise+.005*float(e)
        active_run=active_run+1 if active else 0
        quiet_run=quiet_run+1 if not active else 0
        if not in_speech and active_run>=VAD_START_FRAMES:
            onset=max(0,i-VAD_START_FRAMES+1); in_speech=True; quiet_run=0
        if in_speech:
            # Re-evaluate long continuous speech every 700 ms.
            if (i-onset)*VAD_HOP>=11200 and i-last_emit>=70:
                candidates.append(onset*VAD_HOP); last_emit=i; onset=i-18
            if quiet_run>=VAD_END_FRAMES:
                candidates.append(onset*VAD_HOP); last_emit=i
                in_speech=False; active_run=0
        if not in_speech and not active:
            noise=min(max(noise,VAD_MIN_RMS),.05)
    if in_speech: candidates.append(onset*VAD_HOP)
    starts=[]
    duration=len(x)
    for onset_sample in candidates:
        start=max(0,min(onset_sample-VAD_PRE,duration-N))
        if start>=0 and start+N<=duration:
            if not starts or start-starts[-1]>=int(.45*SR): starts.append(start)
    return np.array(starts,np.int64)

stream_X=[]; stream_meta=[]
for source in active[active.split.eq('development')].itertuples():
    starts=gate_candidates(audio[source.filename])
    begin=len(stream_X)
    for i in range(0,len(starts),128):
        wave=np.stack([
            audio[source.filename][s:s+N].astype(np.float32)/32768.
            for s in starts[i:i+128]])
        stream_X.extend(normalize_v4(frontend(wave)))
    expected=[(float(r.review_start_s),float(r.review_end_s),r.candidate_id)
              for r in words[(words.filename==source.filename)&
                             (words.proposed_label=='vasu')].itertuples()
              if r.candidate_id in set(dev_items.id)]
    stream_meta.append(dict(filename=source.filename,
        category=source.recording_category,speaker=source.speaker,
        starts=starts,expected=expected,begin=begin,end=len(stream_X),
        duration_s=len(audio[source.filename])/SR))
V4_stream_X=np.asarray(stream_X,np.float32)

def gated_metrics(probs,threshold,margin):
    rows=[]
    for ref in stream_meta:
        p=probs[ref['begin']:ref['end']]
        detected=(p[:,0]>=threshold)&((p[:,0]-p[:,1])>=margin)
        times=(ref['starts'][detected]+VAD_PRE)/SR
        used=set(); matched=0; false=0
        for event in times:
            choices=[i for i,(a,b,_) in enumerate(ref['expected'])
                     if i not in used and a-.20<=event<=b+.20]
            if choices: used.add(choices[0]); matched+=1
            else: false+=1
        rows.append(dict(filename=ref['filename'],category=ref['category'],
            speaker=ref['speaker'],candidates=len(p),matched=matched,
            expected=len(ref['expected']),false_events=false,
            duration_s=ref['duration_s']))
    d=pd.DataFrame(rows)
    pos=d[d.category=='keyword']; neg=d[d.category!='keyword']
    by=pos.groupby('speaker')[['matched','expected']].sum()
    recalls=by.matched/by.expected.clip(lower=1)
    return dict(recall=float(pos.matched.sum()/max(1,pos.expected.sum())),
                worst_speaker_recall=float(recalls.min()),
                false_events=int(neg.false_events.sum()),
                false_events_per_min=float(60*neg.false_events.sum()/neg.duration_s.sum()),
                gate_candidates=int(d.candidates.sum())),d

search=[]; winner=None
for _,path,row in keeper.saved:
    candidate=tf.keras.models.load_model(path,compile=False)
    p=predict3(candidate,V4_stream_X)
    for t in (.35,.45,.55,.65,.75,.85,.92,.96):
        for margin in (.0,.10,.20,.30,.40,.50):
            metrics,details=gated_metrics(p,t,margin)
            good=metrics['recall']>=.90 and metrics['worst_speaker_recall']>=.80
            rank=(0 if good else 1,metrics['false_events_per_min'],
                  -metrics['recall'],-metrics['worst_speaker_recall'])
            record=dict(path=path,epoch=row['epoch'],threshold=t,
                        margin=margin,parameters=candidate.count_params(),**metrics)
            search.append(record)
            if winner is None or rank<winner['rank']:
                winner=dict(rank=rank,row=record,details=details)
pd.DataFrame(search).to_csv(RUN/'gated_development_search.csv',index=False)
selected=tf.keras.models.load_model(winner['row']['path'],compile=False)
selected.save(RUN/'selected_float.keras')
winner['details'].to_csv(RUN/'gated_development_by_recording.csv',index=False)
print('\nSELECTED GATED FLOAT DETECTOR')
print(winner['row'])

# Full signed INT8 conversion and re-evaluation.
fixed_in=tf.keras.Input(batch_shape=(1,FRAMES,MELS,1))
fixed=tf.keras.models.clone_model(selected,input_tensors=fixed_in)
fixed.set_weights(selected.get_weights())
cal_rng=np.random.default_rng(SEED+8)
cal=[]
for label in range(3):
    idx=np.flatnonzero(V4_y==label)
    cal.extend(cal_rng.choice(idx,min(120,len(idx)),replace=False))
def representative():
    for i in cal: yield [V4_X[i:i+1].astype(np.float32)]
converter=tf.lite.TFLiteConverter.from_keras_model(fixed)
converter.optimizations=[tf.lite.Optimize.DEFAULT]
converter.representative_dataset=representative
converter.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type=tf.int8; converter.inference_output_type=tf.int8
blob=converter.convert(); (RUN/'vasu_gated_int8.tflite').write_bytes(blob)
interpreter=tf.lite.Interpreter(model_content=blob,num_threads=1)
interpreter.allocate_tensors(); ii=interpreter.get_input_details()[0]; oi=interpreter.get_output_details()[0]
iscale,izero=ii['quantization']; oscale,ozero=oi['quantization']
def int8_probs(X):
    out=[]
    for x in X:
        q=np.clip(np.rint(x/iscale+izero),-128,127).astype(np.int8)
        interpreter.set_tensor(ii['index'],q[None]); interpreter.invoke()
        z=interpreter.get_tensor(oi['index'])[0]
        out.append((z.astype(np.float32)-ozero)*oscale)
    return np.asarray(out)
pi=int8_probs(V4_stream_X)
int8_search=[]; int8_winner=None
for t in (.35,.45,.55,.65,.75,.85,.92,.96):
    for margin in (.0,.10,.20,.30,.40,.50):
        metrics,details=gated_metrics(pi,t,margin)
        good=metrics['recall']>=.90 and metrics['worst_speaker_recall']>=.80
        rank=(0 if good else 1,metrics['false_events_per_min'],
              -metrics['recall'],-metrics['worst_speaker_recall'])
        row=dict(threshold=t,margin=margin,**metrics)
        int8_search.append(row)
        if int8_winner is None or rank<int8_winner['rank']:
            int8_winner=dict(rank=rank,row=row,details=details)
pd.DataFrame(int8_search).to_csv(RUN/'int8_gated_search.csv',index=False)
int8_winner['details'].to_csv(RUN/'int8_gated_by_recording.csv',index=False)
ops=sorted(set(o['op_name'] for o in interpreter._get_ops_details())-{'DELEGATE'})
summary=dict(status='development_selected_not_independently_tested',
    approach='adaptive activity gate plus onset-aligned 3-class DS-CNN',
    parameters=int(selected.count_params()),model_bytes=len(blob),
    model_sha256=hashlib.sha256(blob).hexdigest(),
    threshold=float(int8_winner['row']['threshold']),
    confusable_margin=float(int8_winner['row']['margin']),
    input_scale=float(iscale),input_zero=int(izero),
    output_scale=float(oscale),output_zero=int(ozero),operators=ops,
    int8_development={k:v for k,v in int8_winner['row'].items()
                      if k not in ('threshold','margin')},
    development_target=dict(recall_at_least=.90,
        every_speaker_recall_at_least=.80,
        hard_negative_false_events_per_min_below=1.0),
    development_target_met=bool(int8_winner['row']['recall']>=.90 and
        int8_winner['row']['worst_speaker_recall']>=.80 and
        int8_winner['row']['false_events_per_min']<1.),
    gate=dict(frame_samples=VAD_FRAME,hop_samples=VAD_HOP,
        preroll_samples=VAD_PRE,start_frames=VAD_START_FRAMES,
        end_frames=VAD_END_FRAMES,factor=VAD_FACTOR,min_rms=VAD_MIN_RMS),
    speaker4_used=False,speaker5_used=False,
    hardware_ram_bytes=None,hardware_cpu_percent=None,
    hardware_inference_ms=None,asr_handoff_latency_ms=None)
(RUN/'result_summary.json').write_text(json.dumps(summary,indent=2))
np.savez(RUN/'frontend_constants.npz',hann=window,mel=mel,mean=v4_mean,std=v4_std)
print('\nINT8 GATED DEVELOPMENT RESULT')
print(json.dumps(summary,indent=2))


## 3 — Export ESP handoff and evidence ZIP

In [ ]:

# Export exact model/frontend/gate handoff and one evidence ZIP.
def carr(name,values,ctype='float'):
    flat=np.asarray(values).reshape(-1)
    def f(v):
        if ctype!='float': return str(int(v))
        s=format(float(v),'.9g')
        if '.' not in s and 'e' not in s: s+='.'+'0'
        return s+'f'
    return f'static const {ctype} {name}[{len(flat)}] = {{\n'+',\n'.join(
        ', '.join(f(v) for v in flat[i:i+12]) for i in range(0,len(flat),12))+'\n};\n'
header='#pragma once\n#include <stdint.h>\n'
for key,value in [('VASU_SAMPLE_RATE',SR),('VASU_AUDIO_SAMPLES',N),
                  ('VASU_FRAMES',FRAMES),('VASU_MELS',MELS),
                  ('VASU_VAD_FRAME',VAD_FRAME),('VASU_VAD_HOP',VAD_HOP),
                  ('VASU_VAD_PREROLL',VAD_PRE),
                  ('VASU_VAD_START_FRAMES',VAD_START_FRAMES),
                  ('VASU_VAD_END_FRAMES',VAD_END_FRAMES)]:
    header+=f'#define {key} {value}\n'
header+=f'#define VASU_VAD_FACTOR {VAD_FACTOR}f\n#define VASU_VAD_MIN_RMS {VAD_MIN_RMS}f\n'
header+=carr('vasu_hann',window)+carr('vasu_mel_bin_major',mel)
header+=carr('vasu_feature_mean',v4_mean)+carr('vasu_feature_std',v4_std)
(RUN/'vasu_frontend_and_gate.h').write_text(header)
model_hex=[', '.join(f'0x{b:02x}' for b in blob[i:i+16]) for i in range(0,len(blob),16)]
(RUN/'vasu_model_data.cc').write_text('#include <stdint.h>\nalignas(16) extern const unsigned char g_vasu_model[] = {\n'+',\n'.join(model_hex)+'\n};\nextern const unsigned int g_vasu_model_len=sizeof(g_vasu_model);\n')
(RUN/'vasu_model_data.h').write_text('#pragma once\nextern const unsigned char g_vasu_model[];\nextern const unsigned int g_vasu_model_len;\n')
(RUN/'READ_ME_FIRST.txt').write_text(
    'V4 uses speakers 1-3 only. Speakers 4 and 5 remain sealed.\n'
    'This is development selection, not independent accuracy.\n'
    'Pipeline: adaptive activity gate -> 1.2 s onset-aligned log-Mel -> 3-class INT8 DS-CNN.\n'
    'Hardware RAM, CPU and latency remain unmeasured until ESP deployment.\n')
(RUN/'environment.json').write_text(json.dumps(dict(
    python=sys.version,tensorflow=tf.__version__,numpy=np.__version__,seed=SEED),indent=2))
hist=pd.read_csv(RUN/'training_history.csv')
fig,ax=plt.subplots(1,2,figsize=(10,3.7),constrained_layout=True)
ax[0].plot(hist.epoch,hist.loss,label='train'); ax[0].plot(hist.epoch,hist.val_loss,label='development')
ax[0].set(xlabel='Epoch',ylabel='Loss',title='Onset-aligned 3-class DS-CNN'); ax[0].legend()
t=pd.read_csv(RUN/'int8_gated_search.csv')
for margin,g in t.groupby('margin'):
    ax[1].plot(g.false_events_per_min,g.recall,marker='o',label=f'margin {margin:.1f}')
ax[1].axhline(.9,color='#555',ls=':'); ax[1].set(xlabel='False events/min',ylabel='Keyword recall',title='INT8 gated trade-off'); ax[1].legend(fontsize=7)
fig.savefig(RUN/'development_results.png',dpi=150); plt.show()
archive=Path(shutil.make_archive(str(RUN)+'_results','zip',RUN))
print('RESULT ZIP:',archive)
try:
    from google.colab import files
    files.download(str(archive))
except Exception:
    pass
